In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis, LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, classification_report
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder
import numpy as np
from scipy.sparse import hstack

DATA = Path('data')

## Data Clean
- ??????????
- ??URL?@???#????????????,????????

In [ ]:
# --- 1. ???? ---
print("??????...")
train_df = pd.read_csv(DATA / 'train.csv')
val_df = pd.read_csv(DATA / 'val.csv')
train_df.head()

train_df = train_df.sample(frac=1, random_state=18260817).reset_index(drop=True)
val_df = val_df.sample(frac=1, random_state=18260817).reset_index(drop=True)

In [ ]:
# ???????NaN?
train_df['text'] = train_df['text'].fillna('')
val_df['text'] = val_df['text'].fillna('')
train_df['event'] = train_df['event'].fillna('unknown')
val_df['event'] = val_df['event'].fillna('unknown')

In [ ]:
# --- 2. ??????? ---
def preprocess_text(text):
    """??????"""
    text = text.lower()  # ????
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # ??URL
    text = re.sub(r'\@\w+|\#', '', text) # ??@?#
    text = re.sub(r'[^a-z\s]', '', text) # ??????????
    return text

In [ ]:
print("???????...")
train_df['clean_text'] = train_df['text'].apply(preprocess_text)
val_df['clean_text'] = val_df['text'].apply(preprocess_text)
y_train = train_df['label']
y_val = val_df['label']
train_df.head()

In [ ]:
# --- 2. Feature Engineering (in parallel) ---

# 2.1 Text Path (TF-IDF -> SVD)
print("????????...")
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['clean_text'])
X_val_tfidf = tfidf_vectorizer.transform(val_df['clean_text'])

n_components = 150
svd = TruncatedSVD(n_components=n_components, random_state=42)
X_train_svd = svd.fit_transform(X_train_tfidf)
X_val_svd = svd.transform(X_val_tfidf)


In [ ]:
# # 2.2 Event Path (One-Hot Encoding)
# print("???? 'event' ??...")
# onehot_encoder = OneHotEncoder(handle_unknown='ignore')
# # Note: The input to OneHotEncoder should be a 2D array, hence train_df[['event']]
# X_train_event = onehot_encoder.fit_transform(train_df[['event']])
# X_val_event = onehot_encoder.transform(val_df[['event']])

In [ ]:
# --- 3. Combine Features ---
# print("????????? 'event' ??...")
# # np.hstack requires dense arrays for stacking.
# X_train_combined = np.hstack((X_train_svd, X_train_event.toarray()))
# X_val_combined = np.hstack((X_val_svd, X_val_event.toarray()))



In [ ]:
# print(f"??????: {X_train_combined.shape[1]}") # This will be n_components + number_of_event_categories


In [ ]:
# --- 4. Model Training and Evaluation (QDA) ---

X_train_combined = X_train_svd
X_val_combined = X_val_svd
print("??????? QDA ??...")
qda_model = QuadraticDiscriminantAnalysis(reg_param=1e-5)
# Note: QDA can sometimes fail if a class has too few samples for a given feature combination,
# leading to a singular covariance matrix. If that happens, LDA is a more robust alternative.
try:
    qda_model.fit(X_train_combined, y_train)
    y_pred_qda = qda_model.predict(X_val_combined)
    accuracy_qda = accuracy_score(y_val, y_pred_qda)

    print(f"\n?? 'event' ? QDA ???????????: {accuracy_qda:.4f}")
    print("\nQDA ????:")
    print(classification_report(y_val, y_pred_qda))
except Exception as e:
    print(f"\n????QDA??,???????????: {e}")
    print("?????LDA???????")